# Harmony360 Trainer Reconciliation v1 — Mobile Colab Edition
Designed for phone-based Google Colab use. Small cells, CPU-only verification, progress file, heartbeat during training, and hard timeouts. This is verification-scale training, not an LLM-scale run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted')


In [ ]:
import json, time, subprocess, sys, platform, hashlib
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/HAR360_MASTER/reconciliation/trainer')
ROOT.mkdir(parents=True, exist_ok=True)
PROGRESS_PATH = ROOT / 'MOBILE_PROGRESS.json'
def progress(stage, **details):
    record = {'stage': stage, 'updated_unix': time.time(), **details}
    PROGRESS_PATH.write_text(json.dumps(record, indent=2, sort_keys=True, default=str), encoding='utf-8')
    print('📍', json.dumps(record, indent=2, default=str))
progress('NOTEBOOK_STARTED')


In [ ]:
progress('CLONE_START')
subprocess.run(['bash','-lc','rm -rf /content/Harmony360 && timeout 180s git clone -q --branch reconciliation/trainer-v1 https://github.com/rdntmsn/Harmony360.git /content/Harmony360'], check=True)
progress('CLONE_COMPLETE')
try:
    import torch
    print('✅ PyTorch:', torch.__version__)
except ImportError as exc:
    progress('STOPPED_PYTORCH_MISSING')
    raise RuntimeError('PyTorch missing. Stop here rather than starting a long install.') from exc


In [ ]:
progress('PACKAGE_INSTALL_START')
cmd = 'cd /content/Harmony360/reconciliation/core_runtime_v1 && timeout 480s python -m pip install -q numpy pytest cryptography && timeout 480s python -m pip install -q -e .'
subprocess.run(['bash','-lc',cmd], check=True)
progress('PACKAGE_INSTALL_COMPLETE')
print('✅ Harmony360 runtime installed')


In [ ]:
import numpy, pytest, cryptography, torch
REPO_ROOT = Path('/content/Harmony360/reconciliation/core_runtime_v1')
commit = subprocess.check_output(['git','-C','/content/Harmony360','rev-parse','HEAD'], text=True).strip()
env = {'git_commit': commit, 'python': sys.version.split()[0], 'torch': torch.__version__, 'numpy': numpy.__version__, 'pytest': pytest.__version__, 'cryptography': cryptography.__version__, 'platform': platform.platform()}
progress('ENVIRONMENT_VERIFIED', **env)
print(json.dumps(env, indent=2))


In [ ]:
progress('TEST_SUITE_START')
try:
    result = subprocess.run([sys.executable,'-m','pytest','-q'], cwd=REPO_ROOT, capture_output=True, text=True, timeout=480)
except subprocess.TimeoutExpired:
    progress('TEST_SUITE_TIMEOUT')
    raise RuntimeError('Test suite exceeded 8 minutes.')
print(result.stdout)
if result.stderr: print(result.stderr)
if result.returncode != 0:
    progress('TEST_SUITE_FAILED', returncode=result.returncode)
    raise RuntimeError('Harmony360 runtime tests failed.')
progress('TEST_SUITE_PASS', stdout=result.stdout.strip())


In [ ]:
RUN_DIR = ROOT / 'runs' / 'mobile_verification_train'
RUN_DIR.mkdir(parents=True, exist_ok=True)
TRAINING_SCRIPT = Path('/content/h360_mobile_training_probe.py')
SUMMARY_PATH = RUN_DIR / 'MOBILE_TRAINING_SUMMARY.json'
script = f'''
import json
from pathlib import Path
from harmony360 import Harmony360HybridTrainer, Harmony360TrainingConfig
run_dir = Path({str(RUN_DIR)!r})
summary_path = Path({str(SUMMARY_PATH)!r})
trainer = Harmony360HybridTrainer(run_dir)
trainer.add_text(("Harmony360 training verification uses real parameters, forward passes, cross entropy loss, backpropagation, optimizer updates, snapshots, and ledgers. " * 12), source="mobile_trainer_reconciliation_probe")
config = Harmony360TrainingConfig(sequence_length=12, embedding_dim=12, hidden_dim=24, batch_size=16, epochs=2, learning_rate=0.01, seed=360, device="cpu")
summary = trainer.train_language_model(run_dir / "harmony360_trainer_verification.pt", config=config)
summary_path.write_text(json.dumps(summary, indent=2, sort_keys=True), encoding="utf-8")
print("TRAINING_SUBPROCESS_COMPLETE", flush=True)
'''
TRAINING_SCRIPT.write_text(script, encoding='utf-8')
progress('TRAINING_PREPARED', epochs=2, device='cpu', training_summary_path=str(SUMMARY_PATH))


In [ ]:
progress('TRAINING_START')
started = time.time()
proc = subprocess.Popen([sys.executable, str(TRAINING_SCRIPT)], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
timeout_seconds, heartbeat_seconds, next_heartbeat = 480, 15, 15
while proc.poll() is None:
    elapsed_now = time.time() - started
    if elapsed_now >= timeout_seconds:
        proc.kill(); stdout, stderr = proc.communicate(); print(stdout); print(stderr)
        progress('TRAINING_TIMEOUT', elapsed_seconds=elapsed_now)
        raise RuntimeError('Verification training exceeded 8 minutes and was terminated.')
    if elapsed_now >= next_heartbeat:
        print(f'⏱️ Training still running: {elapsed_now:.0f} seconds')
        progress('TRAINING_HEARTBEAT', elapsed_seconds=elapsed_now)
        next_heartbeat += heartbeat_seconds
    time.sleep(1)
stdout, stderr = proc.communicate(); elapsed = time.time() - started
if stdout: print(stdout)
if stderr: print(stderr)
if proc.returncode != 0: raise RuntimeError('Verification training subprocess failed.')
summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
assert summary['metrics']['parameter_delta_l2'] > 0
progress('TRAINING_COMPLETE', elapsed_seconds=elapsed, final_loss=summary['metrics']['final_loss'], parameter_delta_l2=summary['metrics']['parameter_delta_l2'])
print(json.dumps(summary, indent=2))


In [ ]:
from harmony360 import HAR360LedgerManager, load_trained_language_model
ledger = HAR360LedgerManager(RUN_DIR / 'TRAINING_LEDGER.jsonl')
ledger_verification = ledger.verify_chain()
assert ledger_verification['valid'] is True
model, checkpoint = load_trained_language_model(summary['checkpoint_path'])
assert checkpoint['format'] == 'HARMONY360_TRAINING_CHECKPOINT'
assert checkpoint['model_type'] == 'TinyCausalLanguageModel'
progress('CHECKPOINT_AND_LEDGER_VERIFIED', **ledger_verification)
print('✅ Ledger and checkpoint verified')


In [ ]:
from datetime import datetime, timezone
def sha256_file(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
checkpoint_path = Path(summary['checkpoint_path'])
evidence = {'capability':'Harmony360 Trainer Reconciliation v1','classification':'CORE_RUNTIME_MODEL_TRAINING_RECONCILIATION','status':'PASS','mobile_colab_profile':True,'sealed_utc':datetime.now(timezone.utc).isoformat(),'git':{'repository':'rdntmsn/Harmony360','branch':'reconciliation/trainer-v1','commit_sha':commit,'root_path':'reconciliation/core_runtime_v1'},'runtime':env,'tests':{'command':'python -m pytest -q','returncode':result.returncode,'stdout':result.stdout.strip()},'training_probe':summary,'training_elapsed_seconds':elapsed,'training_ledger_verification':ledger_verification,'checkpoint_sha256_recomputed':sha256_file(checkpoint_path),'scope_of_conclusion':'Verifies actual parameter optimization for a verification-scale causal language model, checkpoint creation/loading, governed training snapshot/ledger linkage, and tested trainer mechanics. It does not establish LLM-scale capability, generalization, or superiority of Harmony360-specific mathematical interventions.'}
canonical = json.dumps(evidence, sort_keys=True, separators=(',',':'), default=str).encode()
evidence['record_sha256'] = hashlib.sha256(canonical).hexdigest()
evidence_path = ROOT / 'TRAINER_RECONCILIATION_v1_SEALED.har360.json'
evidence_path.write_text(json.dumps(evidence, indent=2, sort_keys=True, default=str), encoding='utf-8')
progress('SEALED', evidence_path=str(evidence_path), record_sha256=evidence['record_sha256'])
print('Evidence:', evidence_path)
print('Record SHA256:', evidence['record_sha256'])
